# 🎯 Preset Testing Notebook

This notebook defines and tests multiple preset configurations for comprehensive analysis.
All presets are defined here - the production system only has clean base presets.

## Testing Strategy
- Define presets in notebook for complete control
- Clean production code with only essential presets
- Comprehensive testing of all model + similarity combinations
- Clean logging: INFO for results, DEBUG for details

In [1]:
# System Setup
import os
import sys
import time
import pathlib
import torch
import warnings
warnings.filterwarnings('ignore')

# CRITICAL: Fix Jina wrap_triton import issue BEFORE any model loading
try:
    import torch.library
    if not hasattr(torch.library, 'wrap_triton'):
        torch.library.wrap_triton = lambda fn: fn
        print('✅ Applied wrap_triton fix for Jina models')
    else:
        print('✅ wrap_triton already available')
except Exception as e:
    print(f'⚠️ wrap_triton fix warning: {e}')

# Apply globally to flash_attn as well
try:
    import flash_attn
    if not hasattr(torch.library, 'wrap_triton'):
        torch.library.wrap_triton = lambda fn: fn
        print('✅ Flash attention wrap_triton fix applied')
except ImportError:
    print('📝 flash_attn not installed - fix will be applied when needed')
except Exception as e:
    print(f'⚠️ Flash attention fix warning: {e}')

sys.path.append('/app')
os.chdir('/app')

HF_CACHE = pathlib.Path('models_cache')
HF_CACHE.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)

# Fix potential sentence-transformers import issues
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(HF_CACHE)

from src.logger import get_logger
import logging

log = get_logger(
    name='preset-testing',
    log_dir='logs', 
    console_level=logging.INFO,
    file_level=logging.DEBUG
)

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'

log.info('🎯 PRESET TESTING SYSTEM')
log.info('=' * 50)
log.info(f'🎮 GPU Available: {GPU_AVAILABLE}')
if GPU_AVAILABLE:
    log.info(f'🚀 GPU Device: {torch.cuda.get_device_name(0)}')
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    torch.cuda.empty_cache()
    log.info('✅ GPU configured')

log.info(f'⚙️ Device: {DEVICE}')
log.info('📝 Clean logging: INFO for results, DEBUG for details')
log.info('🔧 Applied Jina compatibility fixes (wrap_triton + custom_st + trust_remote_code)')
log.info('🎯 Ready for comprehensive preset testing!')

✅ Applied wrap_triton fix for Jina models


[09/14/25 22:22:51] INFO     📝 Logging to /app/logs/preset-testing.log.2025-09-14

                    INFO     🎯 PRESET TESTING SYSTEM

                    INFO     ==================================================

                    INFO     🎮 GPU Available: True

...

                    INFO     ✅ GPU configured

                    INFO     ⚙️ Device: cuda

                    INFO     📝 Clean logging: INFO for results, DEBUG for details

                    INFO     🔧 Applied Jina compatibility fixes (wrap_triton + custom_st + trust_remote_code)

                    INFO     🎯 Ready for comprehensive preset testing!

In [2]:
# Initialize System Components
from src.utils import ServiceDatasetLoader
from src.processing import TextNormalizer, MorphReducer, LexicalRelevanceFilter
from src.search import ServiceSearch
from src.logger import init_default_logger

default_log = init_default_logger('naama-search', console_level=logging.INFO, file_level=logging.DEBUG)

log.info('🚀 INITIALIZING SYSTEM')
log.info('=' * 40)

normalizer = TextNormalizer()
morpher = MorphReducer()
lexical_filter = LexicalRelevanceFilter(normalizer)
log.info('✅ Processing components ready')

rename_map = {
    'الاسم عربي': 'service',
    'التصنيف عربي': 'classification', 
    'القطاع عربي': 'sector',
    'الوصف المختصر عربي': 'description_short',
    'الوصف عربي': 'description',
    'المستفيدين من الخدمة': 'beneficiaries',
}

combine_cols = (
    'service', 'service', 'service', 'classification', 
    'sector', 'description_short', 'description', 'beneficiaries'
)

loader_ar = ServiceDatasetLoader(
    'data/NaamaServiceIn full Details.xlsx', 
    rename_map, 
    combine_cols
)

log.info(f'✅ Dataset loaded: {len(loader_ar.documents)} documents')

TEST_QUERIES = [
    'القطط', 'بسة', 'مواشي', 'بيع أعلاف', 'ملكية مزرعة',
    'نقل نحل', 'تربية خيل', 'تربية نحل', 'احفر بير', 'فاكهة', 'خضار'
]

log.info(f'🎯 Test queries: {len(TEST_QUERIES)} queries')
log.info('🎯 System ready for preset testing!')

E0000 00:00:1757888573.550391      68 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757888573.553552      68 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[09/14/25 22:22:55] INFO     📝 Logging to /app/logs/naama-search.log.2025-09-14

[09/14/25 22:22:55] INFO     🚀 INITIALIZING SYSTEM

                    INFO     ========================================

                    INFO     🔒 Protected 12 specific entity terms from heavy normalization

...

                    INFO     📄 Loading [NaamaServiceIn full Details] 'NaamaServiceIn full Details.xlsx'…

                    INFO     ✅ 997 rows → (service, service, service, classification, sector, description_short,  
                             description, beneficiaries)

                    INFO     ✅ Dataset loaded: 997 documents

                    INFO     🎯 Test queries: 11 queries

                    INFO     🎯 System ready for preset testing!

In [3]:
# Define Comprehensive Test Presets - SOTA Models Only
BASE_CONFIG = {
    'data': {
        'ar': {
            'path': './data/NaamaServiceIn full Details.xlsx',
            'rename_map': rename_map,
            'combine_cols': combine_cols,
        }
    },
    'model_config': {
        'trust_remote_code': True  # Required for Jina models
    }
}

def create_test_preset(name, embedding_model, similarity_method,
                      use_morphology=True, use_lexical=False,
                      reranker_model=None, candidates_k=100, alpha=0.7,
                      top_k=15, threshold=50):
    config = {
        **BASE_CONFIG,
        'embedding_model': {
            'ar': embedding_model,
            'en': embedding_model,
            'default': embedding_model
        },
        'similarity': {
            'ar': similarity_method,
            'en': similarity_method,
            'default': similarity_method
        },
        'processing': {
            'use_morphology': use_morphology,
            'use_normalization': True,
            'use_lexical_filtering': use_lexical,
        },
        'search': {
            'top_k': top_k,
            'similarity_threshold_pct': threshold,
            'similarity_threshold': threshold / 100.0,
            'use_lexical_filtering': use_lexical,
        },
        'description': name
    }

    if reranker_model and 'reranker' in similarity_method:
        config['reranker'] = {
            'model': reranker_model,
            'candidates_k': candidates_k,
            'alpha': alpha
        }

    return config

# SOTA Models: Complete set including fixed Jina models and Microsoft
SOTA_MODELS = {
    'jina_v3': 'jinaai/jina-embeddings-v3',
    'jina_v4': 'jinaai/jina-embeddings-v4', 
    'multilingual_e5': 'intfloat/multilingual-e5-large',
    'arabic_bert': 'Omartificial-Intelligence-Space/GATE-AraBert-v1',
    'qwen': 'Qwen/Qwen3-Embedding-0.6B',
    'arabic_qwen': 'Omartificial-Intelligence-Space/Semantic-Ar-Qwen-Embed-0.6B',
    'gemma': 'Omartificial-Intelligence-Space/AraGemma-Embedding-300m',
    'bge_m3': 'BAAI/bge-m3' 
}

# All similarity methods
SIMILARITY_METHODS = ['faiss', 'hnsw', 'scann', 'pure-cosine', 'pure-cosine-reranker']

# Reranker parameters for reranker-based similarities
RERANKER_PARAMS = [
    {'candidates_k': 50, 'alpha': 0.5},
    {'candidates_k': 100, 'alpha': 0.6},
    {'candidates_k': 150, 'alpha': 0.7},
    {'candidates_k': 200, 'alpha': 0.8}
]

# Processing combinations: Test all morphology/lexical combinations
PROCESSING_COMBINATIONS = [
    {'morphology': True, 'lexical': False, 'suffix': 'morph_only'},
    {'morphology': False, 'lexical': False, 'suffix': 'minimal'},
    {'morphology': True, 'lexical': True, 'suffix': 'full_proc'},
    {'morphology': False, 'lexical': True, 'suffix': 'lexical_only'}
]

# Generate comprehensive test presets
TEST_PRESETS = {}

# For each model, similarity method, and processing combination
for model_name, model_path in SOTA_MODELS.items():
    for sim_method in SIMILARITY_METHODS:
        for proc_combo in PROCESSING_COMBINATIONS:
            
            if 'reranker' in sim_method:
                # For reranker methods, create variants with different parameters
                for i, params in enumerate(RERANKER_PARAMS):
                    preset_name = f'{model_name}_{sim_method}_k{params["candidates_k"]}_a{params["alpha"]}_{proc_combo["suffix"]}'
                    description = f'{model_name.replace("_", " ").title()} + {sim_method.replace("-", " ").title()} (k={params["candidates_k"]}, α={params["alpha"]}) - {proc_combo["suffix"].replace("_", " ").title()}'

                    TEST_PRESETS[preset_name] = create_test_preset(
                        description,
                        model_path,
                        sim_method,
                        use_morphology=proc_combo['morphology'],
                        use_lexical=proc_combo['lexical'],
                        reranker_model='oddadmix/arabic-reranker',
                        candidates_k=params['candidates_k'],
                        alpha=params['alpha'],
                        threshold=48 if 'jina' in model_name else 50 if 'multilingual_e5' in model_name or 'bge' in model_name else 55
                    )
            else:
                # For non-reranker methods, create preset with processing combination
                preset_name = f'{model_name}_{sim_method}_{proc_combo["suffix"]}'
                description = f'{model_name.replace("_", " ").title()} + {sim_method.upper()} - {proc_combo["suffix"].replace("_", " ").title()}'

                TEST_PRESETS[preset_name] = create_test_preset(
                    description,
                    model_path,
                    sim_method,
                    use_morphology=proc_combo['morphology'],
                    use_lexical=proc_combo['lexical'],
                    threshold=50 if 'jina' in model_name else 55 if 'multilingual_e5' in model_name or 'bge' in model_name else 60
                )

log.info(f'📋 Comprehensive test presets defined: {len(TEST_PRESETS)} configurations')
log.info(f'🎯 Models: {len(SOTA_MODELS)} ({list(SOTA_MODELS.keys())})')
log.info(f'🔧 Similarity methods: {len(SIMILARITY_METHODS)} ({SIMILARITY_METHODS})')
log.info(f'⚙️ Reranker parameter combinations: {len(RERANKER_PARAMS)}')
log.info(f'🔄 Processing combinations: {len(PROCESSING_COMBINATIONS)} (morph/lexical matrix)')
log.info('✅ Ready for MASSIVE comprehensive testing with ALL SOTA models!')

# Calculate total configurations for verification
non_reranker_configs = len(SOTA_MODELS) * len([s for s in SIMILARITY_METHODS if 'reranker' not in s]) * len(PROCESSING_COMBINATIONS)
reranker_configs = len(SOTA_MODELS) * len([s for s in SIMILARITY_METHODS if 'reranker' in s]) * len(RERANKER_PARAMS) * len(PROCESSING_COMBINATIONS)
total_expected = non_reranker_configs + reranker_configs

log.info(f'📊 Configuration breakdown:')
log.info(f'   • Non-reranker: {non_reranker_configs} configs')
log.info(f'   • Reranker: {reranker_configs} configs') 
log.info(f'   • TOTAL: {total_expected} comprehensive tests')
log.info('🔧 Fixed: Jina v3/v4 with custom_st module + trust_remote_code=True')
log.info('🚀 Includes: Jina v3, Jina v4, Microsoft E5, BGE-M3, Arabic models, Qwen models')

                    INFO     📋 Comprehensive test presets defined: 256 configurations

                    INFO     🎯 Models: 8 (['jina_v3', 'jina_v4', 'multilingual_e5', 'arabic_bert', 'qwen',        
                             'arabic_qwen', 'gemma', 'bge_m3'])

                    INFO     🔧 Similarity methods: 5 (['faiss', 'hnsw', 'scann', 'pure-cosine',                   
                             'pure-cosine-reranker'])

                    INFO     ⚙️ Reranker parameter combinations: 4

                    INFO     🔄 Processing combinations: 4 (morph/lexical matrix)

...

                    INFO        • Non-reranker: 128 configs

                    INFO        • Reranker: 128 configs

                    INFO        • TOTAL: 256 comprehensive tests

                    INFO     🔧 Fixed: Jina v3/v4 with custom_st module + trust_remote_code=True

                    INFO     🚀 Includes: Jina v3, Jina v4, Microsoft E5, BGE-M3, Arabic models, Qwen models

In [4]:
# Testing Function with AGGRESSIVE MEMORY MANAGEMENT
import gc
import psutil
import os

def cleanup_memory():
    """Aggressive memory cleanup"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_memory_usage():
    """Get current memory usage"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # MB

def test_preset(preset_name, config, queries):
    initial_memory = get_memory_usage()
    
    try:
        embedding_model = config['embedding_model']['ar']
        similarity_method = config['similarity']['ar']
        use_morphology = config['processing']['use_morphology']
        use_lexical = config['processing']['use_lexical_filtering']
        
        log.info(f'\\n🧪 Testing: {preset_name}')
        log.info(f'   📦 Model: {embedding_model.split("/")[-1]}')
        log.info(f'   🔧 Similarity: {similarity_method}')
        log.info(f'   🔄 Processing: Morphology={use_morphology}, Lexical={use_lexical}')
        log.info(f'   💾 Memory: {initial_memory:.1f}MB')
        
        # CRITICAL: Cleanup before starting
        cleanup_memory()
        
        start_time = time.time()
        engine = ServiceSearch(
            {'ar': loader_ar},
            config,
            normalizer=normalizer,
            morpher=morpher,
            lexical_filter=lexical_filter
        )
        init_time = time.time() - start_time
        log.info(f'   ⏱️ Init time: {init_time:.1f}s')
        
        query_results = []
        total_time = 0
        total_results = 0
        
        for i, query in enumerate(queries, 1):
            start_query = time.time()
            result = engine.search(query)
            query_time = time.time() - start_query
            
            hits_kept = len(result.get('hits_kept', []))
            top_score = 0
            if result.get('hits_kept'):
                top_score = result['hits_kept'][0].get('final_pct', 0)
            
            # Only log every 3rd query to reduce output
            if i % 3 == 1 or i <= 3 or i >= len(queries) - 1:
                log.info(f'   Q{i:2d}: "{query}" → {hits_kept} results (top: {top_score:.1f}%) [{query_time:.2f}s]')
            
            query_results.append({
                'query': query,
                'hits_kept': hits_kept,
                'query_time': query_time,
                'top_score': top_score
            })
            
            total_time += query_time
            total_results += hits_kept
            
            # Memory cleanup every few queries
            if i % 3 == 0:
                cleanup_memory()
        
        avg_time = total_time / len(queries)
        avg_results = total_results / len(queries)
        avg_score = sum(qr['top_score'] for qr in query_results) / len(query_results)
        
        if avg_score >= 80:
            quality = 'EXCELLENT'
        elif avg_score >= 60:
            quality = 'GOOD'
        elif avg_score >= 40:
            quality = 'MODERATE'
        else:
            quality = 'POOR'
        
        log.info(f'   📊 Summary: {avg_results:.1f} avg results, {avg_score:.1f}% avg score ({quality})')
        log.info(f'   ⏱️ Performance: {avg_time:.3f}s avg query time')
        
        # CRITICAL: Cleanup after test
        del engine
        cleanup_memory()
        
        final_memory = get_memory_usage()
        memory_delta = final_memory - initial_memory
        log.info(f'   💾 Memory: {final_memory:.1f}MB (Δ{memory_delta:+.1f}MB)')
        
        return {
            'preset_name': preset_name,
            'embedding_model': embedding_model,
            'similarity_method': similarity_method,
            'use_morphology': use_morphology,
            'use_lexical': use_lexical,
            'init_time': init_time,
            'avg_query_time': avg_time,
            'avg_results': avg_results,
            'avg_score': avg_score,
            'quality_rating': quality,
            'memory_delta': memory_delta,
            'query_results': query_results,
            'success': True
        }
        
    except Exception as e:
        error_msg = str(e)
        log.error(f'❌ FAILED: {preset_name} - {error_msg[:100]}...')
        
        # CRITICAL: Cleanup even on failure
        cleanup_memory()
        
        return {
            'preset_name': preset_name,
            'error': error_msg,
            'success': False
        }

log.info('🧪 Testing function ready with AGGRESSIVE MEMORY MANAGEMENT!')

                    INFO     🧪 Testing function ready with AGGRESSIVE MEMORY MANAGEMENT!

In [5]:
# RUN COMPREHENSIVE TESTING WITH MEMORY MANAGEMENT
from IPython.display import clear_output

log.info('\\n🚀 COMPREHENSIVE PRESET TESTING WITH MEMORY MANAGEMENT')
log.info('=' * 60)
log.info(f'🎯 Testing {len(TEST_PRESETS)} preset configurations')
log.info(f'🔍 Each with {len(TEST_QUERIES)} queries')
log.info(f'📊 Total operations: {len(TEST_PRESETS) * len(TEST_QUERIES)}')
log.info(f'💾 Memory management: ENABLED')
log.info('⏱️ Estimated time: 3-5 hours')

all_results = []
successful_tests = 0
failed_tests = 0
start_total = time.time()

# Initial memory cleanup
cleanup_memory()
initial_system_memory = get_memory_usage()
log.info(f'💾 Initial system memory: {initial_system_memory:.1f}MB')

for i, (preset_name, config) in enumerate(TEST_PRESETS.items(), 1):
    # Clear notebook output every 20 tests to prevent overflow
    if i % 20 == 0:
        clear_output(wait=True)
        print(f'🔄 Cleared output at test {i}/{len(TEST_PRESETS)} - continuing...')
    
    log.info(f'\\n[{i:3d}/{len(TEST_PRESETS)}] ==========================================')
    
    try:
        # Test with memory monitoring
        current_memory = get_memory_usage()
        if current_memory > initial_system_memory + 2000:  # 2GB increase
            log.warning(f'⚠️ High memory usage: {current_memory:.1f}MB - forcing cleanup')
            cleanup_memory()
        
        result = test_preset(preset_name, config, TEST_QUERIES)
        all_results.append(result)
        
        if result['success']:
            successful_tests += 1
            log.info(f'   ✅ SUCCESS: {preset_name}')
        else:
            failed_tests += 1
            log.error(f'   ❌ FAILED: {preset_name}')
            
    except Exception as e:
        log.error(f'   ❌ CRITICAL: {preset_name} - {str(e)}')
        failed_tests += 1
        all_results.append({
            'preset_name': preset_name,
            'error': f'Critical error: {str(e)}',
            'success': False
        })
        # Force cleanup on critical errors
        cleanup_memory()
    
    # Progress update with memory info
    elapsed = time.time() - start_total
    current_memory = get_memory_usage()
    memory_growth = current_memory - initial_system_memory
    
    if i < len(TEST_PRESETS):
        estimated_total = elapsed * len(TEST_PRESETS) / i
        remaining = estimated_total - elapsed
        log.info(f'   ⏱️ Progress: {i}/{len(TEST_PRESETS)} | Remaining: {remaining/60:.1f}min | Success: {successful_tests}/{i}')
        log.info(f'   💾 Memory: {current_memory:.1f}MB (Growth: +{memory_growth:.1f}MB)')
    
    # Aggressive cleanup every 10 tests
    if i % 10 == 0:
        log.info(f'   🧹 Running memory cleanup at test {i}...')
        cleanup_memory()

# Final results
total_time = time.time() - start_total
final_memory = get_memory_usage()
total_memory_growth = final_memory - initial_system_memory

log.info('\\n🎉 TESTING COMPLETE!')
log.info('=' * 50)
log.info(f'⏱️ Total time: {total_time/60:.1f} minutes')
log.info(f'✅ Successful: {successful_tests}/{len(TEST_PRESETS)}')
log.info(f'❌ Failed: {failed_tests}/{len(TEST_PRESETS)}')
log.info(f'📊 Success rate: {(successful_tests/len(TEST_PRESETS)*100):.1f}%')
log.info(f'💾 Final memory: {final_memory:.1f}MB (Growth: +{total_memory_growth:.1f}MB)')

if successful_tests > 0:
    log.info('\\n🏆 TOP PERFORMING CONFIGURATIONS:')
    successful_results = [r for r in all_results if r['success']]
    successful_results.sort(key=lambda x: x['avg_score'], reverse=True)
    
    for i, result in enumerate(successful_results[:10], 1):
        name = result['preset_name']
        score = result['avg_score']
        results_count = result['avg_results']
        time_per_query = result['avg_query_time']
        log.info(f'   {i:2d}. {name}: {score:.1f}% avg, {results_count:.1f} results, {time_per_query:.3f}s')

log.info('\\n📋 Detailed results available in logs for analysis')
log.info('🎯 Use these results for configuration optimization!')

# Final cleanup
cleanup_memory()

print(f'\\n🎊 COMPREHENSIVE TESTING COMPLETE!')
print(f'📊 {successful_tests}/{len(TEST_PRESETS)} presets successful')
print(f'⏱️ Completed in {total_time/60:.1f} minutes')
print(f'💾 Memory managed: {total_memory_growth:.1f}MB total growth')
print('📝 Check logs for detailed analysis data')

🔄 Cleared output at test 240/256 - continuing...


                    INFO     \n[240/256] ==========================================

                    WARNING  ⚠️ High memory usage: 7112.8MB - forcing cleanup

                    INFO     \n🧪 Testing: bge_m3_pure-cosine_lexical_only

                    INFO        📦 Model: bge-m3

...

                    INFO         9. multilingual_e5_pure-cosine_full_proc: 80.8% avg, 10.2 results, 3.512s

                    INFO        10. multilingual_e5_pure-cosine_lexical_only: 80.8% avg, 10.2 results, 3.339s

                    INFO     \n📋 Detailed results available in logs for analysis

                    INFO     🎯 Use these results for configuration optimization!

\n🎊 COMPREHENSIVE TESTING COMPLETE!
📊 204/256 presets successful
⏱️ Completed in 103.3 minutes
💾 Memory managed: 5947.4MB total growth
📝 Check logs for detailed analysis data


In [6]:
print(f'DONE =========> Completed in {total_time/60:.1f} minutes')

DONE =========> Completed in 103.3 minutes
